# Fabric migration (v2) – step 3: Git snapshot and readiness check

> **AS-IS - independent personal/community project.**
> These notebooks are provided **AS-IS**, without warranty of any kind, express or implied.
> They are a personal/community contribution, not a Microsoft product or service, and are not published on
> behalf of, endorsed, supported or warranted by Microsoft. They do not represent or bind Microsoft and
> create no obligation or commitment by Microsoft. Product names, APIs and documentation are technical
> references only. Use at your own risk and validate in your own environment before production or deletion.

Part of the tenant-migration runbook. **Azure DevOps (Fabric Git integration) is the primary copy of the item
definitions; V2_2's copy in ADLS is the secondary copy and the only copy of the data.**

| Step | Notebook | What it does |
|---|---|---|
| 0 | `V2_0_Permission_Check` | Checks every permission and network path the other notebooks need |
| 1 | `V2_1_Connect_Workspaces_To_Git` | Checks each workspace's Git connection; connects the ones that aren't (dry run first) |
| – | change freeze | Note the time (UTC): `FREEZE_STARTED_UTC` |
| 2 | `V2_2_Backup_Data` | Data + secondary definitions to ADLS (run after the change freeze) |
| **3** | **this notebook** | Checks that every workspace is synced with Git, freezes the synced commit in a locked snapshot branch, saves a raw copy to ADLS, checks every item is saved, writes the migration manifest |
| 4 | `V2_4_Delete_Fabric_Items` | Disconnects Git, deletes the Fabric items (only if step 3 said READY and nothing changed since) |
| – | migration | The workspaces move to the new home region |
| 5 | `V2_5_Restore_From_Git` | Reconnects each workspace to the snapshot and lets Fabric recreate the items from Git |
| 6 | `V2_6_Restore_Data` | Loads the data from the backup run back into the recreated items and checks it (settings printed by step 5) |

**Git sync check first (read-only).** A table shows, per workspace, whether it's **synced** with Git: connected,
nothing uncommitted, nothing incoming, no conflicts (Source control shows 0 changes and 0 updates), with the fix
for each one that isn't. With `REQUIRE_ALL_WORKSPACES_SYNCED = True` (default) nothing is created until **every**
workspace is synced. The notebook doesn't commit for you unless `COMMIT_PENDING_CHANGES = True`.

**What this notebook changes (only with `DRY_RUN = False` and every workspace synced)**
- Only with `COMMIT_PENDING_CHANGES = True`: commits pending workspace changes (additions and modifications;
  deletions block).
- Creates and locks ONE snapshot branch with the same name for all workspaces, `migration-snapshot/<snapshot id>`
  (one per repository, after every workspace's commit), and tags it `migration-snapshot-<snapshot id>`.
- With `SAVE_RAW_COPY_TO_ADLS`: saves every item's raw definition (Fabric API) and a file copy of the snapshot
  commit to `<ADLS_ROOT>/_migration/<snapshot id>/raw/` and `.../git/`. This is the fallback if Azure DevOps is
  unavailable at restore time: V2_5 with `RESTORE_SOURCE = "adls"` recreates the items from `raw/`.
- Workspaces in `NO_GIT_WORKSPACE_IDS` skip the Git checks and get no snapshot branch: their items count as
  saved through the raw copy (plus the V2_2 definition), and V2_5 recreates them from ADLS.
- Writes `manifest.json`, `manifest.sha256`, `items.csv` and a log to `<ADLS_ROOT>/_migration/<snapshot id>/`.

It never changes or deletes Fabric items.

**A workspace is READY only when**
- it's synced with Git (and, by default, so is every other workspace in scope);
- a schema-enabled Lakehouse keeps `defaultSchema` in its Git definition (`lakehouse.metadata.json`), so it comes
  back with schemas;
- every Fabric item is in the snapshot commit (SQL analytics endpoints are recreated with their Lakehouse);
- every in-scope item has an OK raw copy (when `SAVE_RAW_COPY_TO_ADLS`) and an OK definition in the secondary backup, and every Lakehouse, Warehouse and KQL
  database has an OK data backup (SQL endpoints: OK SQL scripts);
- every other Fabric item type is acknowledged in `ACKNOWLEDGED_ITEM_TYPES` (it's deleted with the rest);
- the secondary backup run is complete and started after the change freeze.

**Requirements**
- Fabric administrator, and Contributor or higher on every workspace; your Git credentials configured in Fabric
  (open Source control once in each workspace).
- Workspaces already connected to Azure DevOps by the operator.
- Azure DevOps personal access token in Key Vault, scope *Code (Read, write & manage)*, with *Create branch* and
  *Contribute* on the repository (locking a branch needs Contribute).
- *Storage Blob Data Contributor* on the backup container.

In [ ]:
# ---------------------------------------------------------------- scope
# Workspace IDs to migrate (from the inventory workbook). Each must already be connected to Azure DevOps.
WORKSPACE_IDS = [
    # "00000000-0000-0000-0000-000000000000",
]
# In-scope Fabric item types: saved in Git AND in the secondary ADLS backup.
ITEM_TYPES = ["Dataflow", "DataAgent", "Eventhouse", "KQLDatabase", "GraphModel", "Lakehouse", "Ontology",
              "SQLEndpoint", "Warehouse", "DataPipeline", "Notebook"]
# Other Fabric item types found in the workspaces are deleted in the migration too. List a type here only after
# you've decided how it's recovered (for example: definition from Git, data rebuilt by its source). Until then
# the workspace stays BLOCKED.
ACKNOWLEDGED_ITEM_TYPES = []

# ---------------------------------------------------------------- Git (primary copy)
# Organization of the workspaces' Azure DevOps connections, and the token used to read the repository and create
# the snapshot branches.
ADO_ORGANIZATION = "<devops-organization>"
KEY_VAULT_URL = "https://<key-vault-name>.vault.azure.net/"
ADO_AUTH = "pat"                         # how notebooks sign in to Azure DevOps: "pat" or "service_principal"
ADO_PAT_SECRET_NAME = "<secret-name>"    # pat: Key Vault secret holding the personal access token
ADO_SP_TENANT_ID = ""                    # service_principal: Entra tenant ID (GUID) of the app registration
ADO_SP_CLIENT_ID = ""                    # service_principal: application (client) ID
ADO_SP_SECRET_NAME = ""                  # service_principal: Key Vault secret holding the client secret VALUE
# One snapshot branch with the SAME name for every workspace: <prefix>/<snapshot id> (one per repository), at the
# branch head after all commits, plus an annotated tag <prefix>-<snapshot id> on the same commit.
SNAPSHOT_BRANCH_PREFIX = "migration-snapshot"
CREATE_SNAPSHOT_TAG = True
LOCK_SNAPSHOT_BRANCHES = True                    # locked branches can't be changed by anyone without unlocking
# Git sync check: a workspace is synced when it's connected, nothing is uncommitted in the workspace, nothing is
# incoming from Git and there are no conflicts (Source control shows 0 changes and 0 updates).
# True = the snapshot is taken only when EVERY workspace is synced; otherwise nothing is created (no commit, no
# branch, no raw copy, no manifest) and the sync table says what to do in each workspace.
REQUIRE_ALL_WORKSPACES_SYNCED = True
# False = this notebook never commits: commit in each workspace's Source control. True = it commits uncommitted
# workspace changes for you, only in a real run and only when nothing else is out of sync.
COMMIT_PENDING_CHANGES = False
# A workspace change of type "Deleted" means the item exists in Git but not in the workspace. Committing it would
# remove it from Git. Blocked unless you set this to True after confirming the item was deleted on purpose.
ALLOW_COMMITTING_DELETIONS = False

# ---------------------------------------------------------------- secondary backup (ADLS, from V2_2_Backup_Data)
ADLS_ACCOUNT = "<storage-account-name>"
ADLS_CONTAINER = "fabric-backup"
ADLS_ROOT = "fabric-backups"
SECONDARY_BACKUP_RUN_ID = "<run id, e.g. 20261120T190000Z>"   # the backup run taken after the change freeze
# When the change freeze started (UTC, ISO 8601, e.g. "2026-11-20T18:00:00Z"). The backup run must start after it.
FREEZE_STARTED_UTC = ""
# Items in a Git-linked workspace that Git doesn't hold (for example item types Fabric Git doesn't support) count as
# covered when True: V2_5 recreates them from their definition in ADLS (raw copy of this snapshot, else the V2_2
# backup) after the Git update, and V2_4 checks they didn't change since the snapshot. Needs
# SAVE_RAW_COPY_TO_ADLS = True. False = such items block the workspace.
ALLOW_SECONDARY_ONLY_ITEMS = False

# ---------------------------------------------------------------- workspaces without Git
# Workspaces that stay WITHOUT a Git link (it can't or shouldn't be made). They skip the Git checks; their items are
# recreated by V2_5 from the raw definitions saved with this snapshot in ADLS, and V2_4 proves nothing changed by
# comparing each item's definition with that copy. Needs SAVE_RAW_COPY_TO_ADLS = True.
NO_GIT_WORKSPACE_IDS = []

# ---------------------------------------------------------------- raw copy (fallback if Azure DevOps has a problem)
# True = also save, at the moment of the snapshot, every item's raw definition from the Fabric API and a file copy
# of the snapshot commit, under <ADLS_ROOT>/_migration/<snapshot id>/raw/ and .../git/. The raw/ folder uses the
# backup layout; V2_5 recreates items from it when Azure DevOps is unavailable (RESTORE_SOURCE = "adls"), for
# workspaces without Git, and for items Git doesn't hold.
SAVE_RAW_COPY_TO_ADLS = True

# ---------------------------------------------------------------- safety
# True = read and report only: no commit, no branch, no manifest.
DRY_RUN = True

## Helpers
Shared by the v2 notebooks: Fabric REST with retries and long-running operations, ADLS, Azure DevOps Git and
Fabric Git integration calls. Nothing runs here.

In [ ]:
BUILD_ID = "6c473ade12be"
print("V2_3_Git_Snapshot build 6c473ade12be (compare with BUILD_IDS.txt of the package you meant to run)")
# AS-IS, without warranty of any kind; independent personal/community code.
# Not a Microsoft product or service; not published on behalf of Microsoft.
# Does not represent or bind Microsoft; no Microsoft endorsement, warranty or support commitment.
"""Credential redaction for diagnostic output, never for API or backup payloads."""
import re
from collections.abc import Mapping
from typing import TypeVar, overload
from urllib.parse import quote

REDACTED = "<redacted>"
if "_REDACTION_VALUES" not in globals():
    _REDACTION_VALUES: set[str] = set()

_REDACTION_KEYS = {
    "authorization", "accesstoken", "refreshtoken", "clientsecret", "serviceprincipalsecret",
    "password", "passwd", "accountkey", "apikey", "personalaccesstoken", "pat", "secret", "token",
    "connectionstring", "sastoken", "sharedaccesssignature", "sig",
}
_REDACTION_PATTERNS = (
    re.compile(r"\bgh[pousr]_[A-Za-z0-9]{20,}\b"),
    re.compile(r"\bgithub_pat_[A-Za-z0-9_]{40,}\b"),
    re.compile(r"\beyJ[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\b"),
    re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----.*?"
               r"-----END (?:RSA |EC |OPENSSH )?PRIVATE KEY-----", re.DOTALL),
)
_REDACTION_AUTH = re.compile(r"\b(Bearer|Basic)([ \t]+)[A-Za-z0-9_+/=.-]{16,}", re.IGNORECASE)
_REDACTION_QUERY = re.compile(
    r"([?&](?:sig|access_token|refresh_token|client_secret|password|api_key)=)[^&\s\"'<>]+",
    re.IGNORECASE,
)
_REDACTION_ASSIGNMENT = re.compile(
    r"""(\b(?:client[_-]?secret|servicePrincipalSecret|access[_-]?token|refresh[_-]?token|
    password|passwd|account[_-]?key|api[_-]?key|personalAccessToken|sas[_-]?token|pat|secret|token)
    \b["']?\s*[:=]\s*["']?)([^\s"'&,;}\]<>]+)""",
    re.IGNORECASE | re.VERBOSE,
)


def register_sensitive_value(value: str) -> None:
    if len(value) >= 8:
        _REDACTION_VALUES.add(value)


def redact_sensitive_text(text: str) -> str:
    for value in sorted(_REDACTION_VALUES.copy(), key=len, reverse=True):
        text = text.replace(value, REDACTED).replace(quote(value, safe=""), REDACTED)
    for pattern in _REDACTION_PATTERNS:
        text = pattern.sub(REDACTED, text)
    text = _REDACTION_AUTH.sub(lambda m: m[1] + m[2] + REDACTED, text)
    text = _REDACTION_QUERY.sub(lambda m: m[1] + REDACTED, text)
    return _REDACTION_ASSIGNMENT.sub(lambda m: m[1] + REDACTED, text)


_Key = TypeVar("_Key")


@overload
def redact_sensitive_data(value: str) -> str: ...


@overload
def redact_sensitive_data(value: Mapping[_Key, object]) -> dict[_Key, object]: ...


@overload
def redact_sensitive_data(value: list[object]) -> list[object]: ...


@overload
def redact_sensitive_data(value: tuple[object, ...]) -> tuple[object, ...]: ...


@overload
def redact_sensitive_data(value: object) -> object: ...


def redact_sensitive_data(value: object) -> object:
    if isinstance(value, Mapping):
        out = {}
        for key, item in value.items():
            normalized = re.sub(r"[^a-z0-9]", "", str(key).lower())
            if normalized in _REDACTION_KEYS and item is not None and not (isinstance(item, str) and item == ""):
                out[key] = REDACTED
            else:
                out[key] = redact_sensitive_data(item)
        return out
    if isinstance(value, list):
        return [redact_sensitive_data(item) for item in value]
    if isinstance(value, tuple):
        return tuple(redact_sensitive_data(item) for item in value)
    return redact_sensitive_text(value) if isinstance(value, str) else value
import base64, datetime as dt, hashlib, json, re, threading, time, uuid
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote

import pandas as pd

pd.set_option("display.max_colwidth", None)   # never cut error messages or fixes in tables
import requests

try:
    notebookutils  # preloaded in Fabric notebooks
except NameError:
    import notebookutils

FABRIC_API = "https://api.fabric.microsoft.com/v1"
ONELAKE_DFS = "https://onelake.dfs.fabric.microsoft.com"
STORAGE_VERSION = "2023-11-03"
MAX_ATTEMPTS = 6
MAX_RETRY_WAIT_SECONDS = 15 * 60
LRO_TIMEOUT_SECONDS = 60 * 60
CHUNK_BYTES = 8 * 1024 * 1024
ZERO_GUID = "00000000-0000-0000-0000-000000000000"
GUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$")
NOW_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

RESULTS = []           # one row per item and step: this notebook's log
_lock = threading.Lock()
CALLS, RETRIES = Counter(), Counter()


def norm_id(value):
    text = str(value or "").strip().strip("{}").lower()
    return text if GUID_RE.match(text) else None


def safe_name(name, max_len=80):
    """A folder-safe version of a Fabric display name."""
    text = re.sub(r'[\\/:*?"<>|#%\x00-\x1f]', "_", str(name or "")).strip().strip(".")
    return (text[:max_len].rstrip(" .") or "unnamed")


def log(workspace, item, step, status, detail=""):
    """One row of this notebook's log. workspace: name; item: dict with type/displayName/id (or {})."""
    detail = redact_sensitive_text(str(detail))
    with _lock:
        RESULTS.append({"Workspace": workspace, "Item": item.get("displayName", ""), "Type": item.get("type", ""),
                        "ItemId": item.get("id", ""), "Step": step, "Status": status,
                        "Detail": str(detail)})
    if status not in ("OK", "Dry run", "Skipped", "Unchanged", "Info", "Ready"):
        print(f"  ! {workspace} / {item.get('type', '')} {item.get('displayName', '')} / {step}: {status} - "
              f"{detail}")


def show(df, rows=200):
    try:
        display(df.head(rows))
    except NameError:
        print(df.head(rows).to_string())


def human(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024


class HttpError(Exception):
    def __init__(self, status, code, message, url):
        self.status, self.code, self.message, self.url = status, code or "", redact_sensitive_text(str(message or "")), url
        super().__init__(self.short())

    def short(self):
        head = f"HTTP {self.status} {self.code}".strip() if self.status else str(self.code)
        return f"{head}: {self.message}" if self.message else head


RAW_BODY_LIMIT = 4000   # only for non-JSON bodies (for example a whole HTML page); JSON error messages are kept whole


def _raw_text(text):
    text = re.sub(r"<[^>]+>", " ", text or "")              # HTML page -> its text
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= RAW_BODY_LIMIT:
        return text
    return text[:RAW_BODY_LIMIT] + f" ... [{len(text) - RAW_BODY_LIMIT} more characters of the response body]"


def _error_details(resp):
    try:
        body = resp.json()
    except ValueError:
        return "", _raw_text(resp.text)
    if not isinstance(body, dict):
        return "", json.dumps(body)
    err = body.get("error") if isinstance(body.get("error"), dict) else body
    code = err.get("errorCode") or err.get("code") or err.get("typeKey") or err.get("error") or ""
    msg = err.get("message") or err.get("@message") or err.get("error_description") or ""
    if not msg:
        msg = json.dumps(body)                                  # unknown error shape: keep all of it
    return str(code), str(msg)


def _retry_after(resp, attempt):
    try:
        return max(1, int(float(resp.headers.get("Retry-After", ""))))
    except (TypeError, ValueError):
        return min(300, 10 * 2 ** (attempt - 1))


_local = threading.local()


def _session():
    if getattr(_local, "session", None) is None:
        _local.session = requests.Session()
    return _local.session


def http(method, url, audience=None, *, auth=None, headers=None, params=None, json_body=None, data=None,
         timeout=300, retry=True, label="other"):
    """One HTTP call with a fresh Entra token. Retries 408/429/5xx (honouring Retry-After) and network errors."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        hdrs = dict(headers or {})
        if auth:
            hdrs["Authorization"] = auth
        elif audience:
            hdrs["Authorization"] = "Bearer " + notebookutils.credentials.getToken(audience)
        if hdrs.get("Authorization"):
            register_sensitive_value(hdrs["Authorization"].split(" ", 1)[-1])
        try:
            resp = _session().request(method, url, params=params, json=json_body, data=data,
                                      headers=hdrs, timeout=timeout)
        except requests.RequestException as exc:
            if not retry or attempt == MAX_ATTEMPTS:
                raise HttpError(None, type(exc).__name__, str(exc), url)
            time.sleep(min(60, 5 * attempt))
            continue
        with _lock:
            CALLS[label] += 1
        if retry and (resp.status_code in (408, 429) or resp.status_code >= 500):
            wait = _retry_after(resp, attempt)
            if attempt < MAX_ATTEMPTS and wait <= MAX_RETRY_WAIT_SECONDS:
                with _lock:
                    RETRIES[label] += 1
                time.sleep(wait)
                continue
        if resp.status_code >= 400:
            raise HttpError(resp.status_code, *_error_details(resp), url)
        return resp


# ---------------------------------------------------------------- Fabric REST API
def _wait_operation(resp, want_result=True):
    """Follow a Fabric long-running operation (202 Accepted) to its result."""
    location = resp.headers.get("Location")
    op_id = resp.headers.get("x-ms-operation-id")
    if not location and op_id:
        location = f"{FABRIC_API}/operations/{op_id}"
    if not location:
        return {}
    wait = _retry_after(resp, 1) if resp.headers.get("Retry-After") else 2
    deadline = time.monotonic() + LRO_TIMEOUT_SECONDS
    while True:
        time.sleep(min(max(wait, 1), 30))
        state = http("GET", location, "pbi", label="fabric").json()
        status = str(state.get("status", "")).lower()
        if status == "succeeded":
            if not want_result:
                return state
            try:
                result = http("GET", location.rstrip("/") + "/result", "pbi", label="fabric")
            except HttpError as exc:
                if exc.status in (400, 404):      # operations without a result (commit, update from Git)
                    return state
                raise
            return result.json() if result.content else {}
        if status in ("failed", "cancelled", "canceled"):
            err = state.get("error") or {}
            raise HttpError(None, err.get("errorCode") or status, err.get("message"), location)
        if time.monotonic() > deadline:
            raise HttpError(None, "Timeout", "the long-running operation didn't finish", location)
        wait = 3


def fabric(method, path, body=None, params=None, want_result=True):
    resp = http(method, FABRIC_API + path, "pbi", json_body=body, params=params, label="fabric")
    if resp.status_code == 202:
        return _wait_operation(resp, want_result)
    return resp.json() if resp.content else {}


def fabric_pages(path, key="value", params=None):
    url, prm, seen = FABRIC_API + path, dict(params or {}), set()
    while url:
        body = http("GET", url, "pbi", params=prm or None, label="fabric").json()
        yield from body.get(key) or []
        if body.get("continuationUri"):
            url, prm = body["continuationUri"], {}
        elif body.get("continuationToken"):
            url, prm = FABRIC_API + path, {**(params or {}), "continuationToken": body["continuationToken"]}
        else:
            break
        marker = (url, json.dumps(prm, sort_keys=True))
        if marker in seen:
            break
        seen.add(marker)


# ---------------------------------------------------------------- ADLS Gen2 (secondary backup + migration records)
ADLS_HEADERS = {"x-ms-version": STORAGE_VERSION}


def adls_url(path):
    return f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}/{quote(path.strip('/'), safe='/')}"


def adls_create(path):
    http("PUT", adls_url(path), "storage", headers=ADLS_HEADERS, params={"resource": "file"}, label="adls")


def adls_append(path, position, data):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=data,
         params={"action": "append", "position": str(position)}, label="adls")


def adls_flush(path, position):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=b"",
         params={"action": "flush", "position": str(position)}, label="adls")


def adls_write(path, data):
    """Write a whole (small) file, replacing any existing one."""
    if isinstance(data, str):
        data = data.encode("utf-8")
    adls_create(path)
    for pos in range(0, len(data), CHUNK_BYTES):
        adls_append(path, pos, data[pos:pos + CHUNK_BYTES])
    adls_flush(path, len(data))
    return len(data)


def adls_list(path, recursive=True):
    """[(path relative to `path`, size, is_dir)] under a folder of the container; [] if it doesn't exist."""
    url = f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}"
    params = {"resource": "filesystem", "recursive": str(recursive).lower(), "directory": path.strip("/")}
    out, prefix = [], path.strip("/") + "/"
    while True:
        try:
            resp = http("GET", url, "storage", headers=ADLS_HEADERS, params=params, label="adls")
        except HttpError as exc:
            if exc.status == 404:
                return out
            raise
        for e in resp.json().get("paths", []):
            name = e["name"][len(prefix):] if e["name"].startswith(prefix) else e["name"]
            out.append((name, int(e.get("contentLength") or 0), str(e.get("isDirectory", "false")).lower() == "true"))
        token = resp.headers.get("x-ms-continuation")
        if not token:
            return out
        params = {**params, "continuation": token}


def adls_read(path):
    return http("GET", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls").content


def adls_delete(path):
    http("DELETE", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls")


def adls_exists(path):
    try:
        http("HEAD", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls")
        return True
    except HttpError as exc:
        if exc.status == 404:
            return False
        raise


# ---------------------------------------------------------------- Power BI item types
# Power BI items survive a cross-region move; every other item type is a Fabric item that must be deleted first.
POWER_BI_TYPES = {"Report", "SemanticModel", "Dashboard", "PaginatedReport"}
# Items Fabric creates and deletes together with their parent; they are never saved in Git on their own.
CHILD_TYPES = {"SQLEndpoint"}
DATA_TYPES = {"Lakehouse": "Data", "Warehouse": "Data", "KQLDatabase": "KQL export"}
# Dataflow Gen2 staging items: Fabric creates them for Dataflows Gen2 (one Lakehouse and one Warehouse per workspace,
# named StagingLakehouseForDataflows_<timestamp> / StagingWarehouseForDataflows_<timestamp>) and creates them again
# when a Dataflow Gen2 runs. Not saved in Git, not backed up, not restored; V2_4 still deletes them (a Fabric item
# left behind blocks a tenant migration).
STAGING_RE = re.compile(r"^Staging(Lakehouse|Warehouse)ForDataflows(_\d+)?$", re.IGNORECASE)


def is_dataflow_staging(item):
    return bool(item.get("staging")) or (item.get("type") in ("Lakehouse", "Warehouse", "SQLEndpoint")
                                         and bool(STAGING_RE.match(str(item.get("displayName") or ""))))
KQL_ADX_STEP = "Kusto Copy"      # V2_2 with KQL_BACKUP_MODE "adx"/"both": the KQL rows are in Azure Data Explorer


def data_step(item_type, status_of):
    """(backup step, status) holding an item's data. KQL databases: the Parquet export if it's OK, otherwise the
    Kusto Copy to ADX if that's OK, otherwise whichever the backup recorded. status_of(step) -> status or ""."""
    step = DATA_TYPES.get(item_type)
    if item_type != "KQLDatabase":
        return step, status_of(step) if step else ""
    exp, adx = status_of(step), status_of(KQL_ADX_STEP)
    if exp == "OK" or not adx:
        return (step, exp) if exp or not adx else (KQL_ADX_STEP, adx)
    if adx == "OK" or not exp:
        return KQL_ADX_STEP, adx
    return step, exp


GUID_ANY_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")


def definition_hash(parts):
    """Fingerprint of an item definition (every part except .platform), to prove it didn't change."""
    h = hashlib.sha256()
    for p in sorted(parts, key=lambda x: x["path"]):
        if p["path"] == ".platform":
            continue
        h.update(p["path"].encode("utf-8") + b"\0" + base64.b64decode(p.get("payload") or "") + b"\0")
    return h.hexdigest()


def adls_only(item):
    """Item in a Git-linked workspace that Git doesn't hold: V2_5 recreates it from its ADLS definition.
    ("secondary backup ..." is the label older snapshots used.)"""
    rec = str(item.get("recovery") or "")
    return not item.get("inGit") and (rec.startswith("ADLS definition (V2_5") or rec.startswith("secondary backup"))


def key_of(item_type, name):
    return f"{item_type}|{str(name).strip().lower()}"


# ---------------------------------------------------------------- Azure DevOps (read the snapshot, manage branches)
ADO_API_VERSION = "7.1"
_ado_auth = {}


ADO_RESOURCE = "499b84ac-1321-427f-aa17-267ca6975798"     # Azure DevOps' fixed Entra application ID


def ado_mode():
    mode = str(globals().get("ADO_AUTH", "pat") or "pat").strip().lower().replace("-", "_").replace(" ", "_")
    if mode in ("sp", "spn", "serviceprincipal"):
        mode = "service_principal"
    if mode not in ("pat", "service_principal"):
        raise ValueError(f"ADO_AUTH = {globals().get('ADO_AUTH')!r}: use \"pat\" or \"service_principal\"")
    return mode


def ado_auth_missing():
    """Settings the chosen ADO_AUTH needs that are still empty or placeholders."""
    need = ["ADO_PAT_SECRET_NAME"] if ado_mode() == "pat" else ["ADO_SP_TENANT_ID", "ADO_SP_CLIENT_ID", "ADO_SP_SECRET_NAME"]
    return [n for n in need if not str(globals().get(n) or "").strip() or "<" in str(globals().get(n))]


def _clean_secret(value):
    value = str(value or "").strip().strip('"').strip("'").strip()
    register_sensitive_value(value)
    return value


_sp_auth = {}


def sp_header():
    """Entra token (Bearer header) of the service principal in ADO_SP_* for Azure DevOps, renewed 5 minutes before it
    expires. Used by ADO_AUTH = "service_principal" and to check GIT_LINK_AUTH = "service_principal"."""
    if "header" in _sp_auth and _sp_auth.get("expires", 0) > time.time() + 300:
        return _sp_auth["header"]
    secret = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_SP_SECRET_NAME))
    kind, _ = secret_shape(secret)
    _sp_auth.update(secret_length=len(secret), secret_kind=kind)
    url = f"https://login.microsoftonline.com/{quote(str(ADO_SP_TENANT_ID).strip())}/oauth2/v2.0/token"
    try:
        tok = http("POST", url, data={"grant_type": "client_credentials", "client_id": str(ADO_SP_CLIENT_ID).strip(),
                                      "client_secret": secret, "scope": ADO_RESOURCE + "/.default"},
                   label="entra").json()
    except HttpError as exc:
        raise HttpError(exc.status, exc.code or "SignInFailed",
                        f"service principal sign-in failed: {exc.message} {sp_signin_fix(exc.message, kind)}".strip(),
                        url) from None
    _sp_auth["header"] = "Bearer " + tok["access_token"]
    register_sensitive_value(tok["access_token"])
    _sp_auth["expires"] = time.time() + int(tok.get("expires_in", 3600))
    return _sp_auth["header"]


def ado_header():
    """Authorization header for Azure DevOps: a personal access token, or an Entra token of a service principal
    (renewed 5 minutes before it expires, so long runs keep working). Secrets come from Key Vault and are never shown."""
    if ado_mode() == "pat":
        if "header" not in _ado_auth:
            pat = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_PAT_SECRET_NAME))
            _ado_auth.update(mode="pat", pat_length=len(pat), pat_shape=secret_shape(pat))
            _ado_auth["header"] = "Basic " + base64.b64encode((":" + pat).encode()).decode()
        return _ado_auth["header"]
    _ado_auth["mode"] = "service_principal"
    try:
        header = sp_header()
    finally:
        _ado_auth.update(pat_length=_sp_auth.get("secret_length"), sp_secret_kind=_sp_auth.get("secret_kind"))
    _ado_auth.update(header=header, expires=_sp_auth["expires"])
    return header


def sp_signin_fix(message, kind=""):
    m = str(message or "")
    if kind == "pat":
        return ("The Key Vault secret looks like a personal access token, not a client secret: set ADO_AUTH = \"pat\" "
                "or store the app's client secret value.")
    if "AADSTS7000215" in m or "AADSTS7000222" in m:
        return ("Fix: the client secret is wrong or expired. Store the secret's Value (not its Secret ID) from the app "
                "registration > Certificates & secrets, as a new version of the Key Vault secret.")
    if "AADSTS700016" in m:
        return "Fix: ADO_SP_CLIENT_ID isn't an app in tenant ADO_SP_TENANT_ID: check both IDs."
    if "AADSTS90002" in m or "AADSTS900023" in m:
        return "Fix: ADO_SP_TENANT_ID is wrong: use the Directory (tenant) ID from the app registration's overview."
    return "Check ADO_SP_TENANT_ID, ADO_SP_CLIENT_ID and the client secret value."


def secret_shape(value):
    """(kind, hint) describing what a secret looks like, without revealing it. kind 'pat' = plausible DevOps token."""
    v, n = value, len(value)
    if not v:
        return "empty", "The secret is empty: paste the token as the secret value."
    if re.fullmatch(r"[a-z2-7]{52}", v) or (n == 84 and re.fullmatch(r"[A-Za-z0-9]{84}", v)):
        return "pat", ""
    if re.match(r"^[A-Za-z0-9_\-.~]{3}\dQ~", v) or (n == 40 and "~" in v):
        return "client secret", ("It looks like an Entra ID app registration client secret (40 characters with 'Q~'), "
                                 "not an Azure DevOps personal access token. Create a token in Azure DevOps (User "
                                 "settings > Personal access tokens, scope Code: Read & write) and store that value.")
    if re.fullmatch(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", v):
        return "GUID", ("It is a GUID: probably a client ID, tenant ID or a client secret's 'Secret ID'. None of "
                        "these is an Azure DevOps token.")
    if v.startswith(("ghp_", "gho_", "ghs_", "github_pat_")):
        return "GitHub token", "It looks like a GitHub token, not an Azure DevOps personal access token."
    if re.fullmatch(r"[0-9a-fA-F]{40}", v):
        return "40 hex", ("It is 40 hexadecimal characters: a Git commit ID or an old GitHub token, not an Azure "
                          "DevOps personal access token.")
    if v.startswith(("{", "[")):
        return "JSON", "It holds a JSON value: store only the token itself."
    if v.lower().startswith(("http://", "https://")):
        return "URL", "It holds a URL: store the token, and put the organization in ADO_ORGANIZATION."
    if v.lower().startswith(("bearer ", "basic ")) or re.search(r"\s", v):
        return "several values", "It holds more than one value (spaces or a 'Bearer'/'Basic' prefix): store only the token."
    if v.startswith("eyJ"):
        return "JWT", "It is an Entra access token (JWT), which expires within hours: store a personal access token."
    return "unknown", (f"Azure DevOps tokens have 52 or 84 characters; this has {n}. Check that the secret holds "
                       "the token and nothing else.")


def ado_org_name(value):
    """'contoso', 'https://dev.azure.com/contoso/...' or 'https://contoso.visualstudio.com' -> 'contoso'."""
    text = str(value or "").strip().rstrip("/")
    m = re.match(r"^(?:https?://)?dev\.azure\.com/([^/]+)", text, re.I) or \
        re.match(r"^(?:https?://)?([^./]+)\.visualstudio\.com", text, re.I)
    return m.group(1) if m else text


SIGN_IN_PAGE = ("Azure DevOps answered with its sign-in page instead of data (HTTP 203): it didn't accept the "
                "personal access token for this organization")


SP_NOT_IN_ORG = ("Add the service principal to the organization (Organization settings > Users > Add users, "
                 "search the app name, Basic access) and to the project's Contributors group; the organization must "
                 "be connected to the app's Entra tenant.")


def ado_call(url, method="GET", params=None, body=None, raw=False, auth=None):
    """auth = another Authorization header (for example sp_header()); default: the ADO_AUTH sign-in."""
    sp = auth is not None or _ado_auth.get("mode") == "service_principal"
    try:
        resp = http(method, url, auth=auth or ado_header(), params={"api-version": ADO_API_VERSION, **(params or {})},
                    json_body=body, label="devops", headers={"Accept": "application/octet-stream"} if raw else None)
    except HttpError as exc:
        if exc.status == 401 and (sp or _ado_auth.get("mode") == "service_principal"):
            raise HttpError(401, exc.code or "Unauthorized", f"{exc.message} {SP_NOT_IN_ORG}".strip(), url) from None
        raise
    ctype = resp.headers.get("Content-Type", "json")
    if resp.status_code == 203 or "html" in ctype:
        if sp or _ado_auth.get("mode") == "service_principal":
            raise HttpError(203, "SignInPage", "Azure DevOps answered with its sign-in page instead of data (HTTP "
                            "203): it didn't accept the service principal's token. " + SP_NOT_IN_ORG, url)
        kind, hint = _ado_auth.get("pat_shape", ("pat", ""))
        raise HttpError(203, "SignInPage", SIGN_IN_PAGE + (f". {hint}" if kind != "pat" else ""), url)
    if raw:
        return resp.content
    if "json" not in ctype:
        raise HttpError(resp.status_code, "NotJson", f"Azure DevOps returned {ctype}, not JSON", url)
    return resp.json() if resp.content else {}


def ado(gp, method, path, params=None, body=None, raw=False):
    """Call the Azure DevOps Git API of the repository in a Fabric Git connection (gp = gitProviderDetails)."""
    url = (f"https://dev.azure.com/{quote(ado_org_name(gp['organizationName']))}/{quote(gp['projectName'])}"
           f"/_apis/git/repositories/{quote(gp['repositoryName'])}{path}")
    return ado_call(url, method, params, body, raw)


def ado_ref(gp, branch):
    """{'objectId', 'isLocked'} of a branch, or None if it doesn't exist."""
    refs = ado(gp, "GET", "/refs", params={"filter": f"heads/{branch}"}).get("value", [])
    return next((r for r in refs if r.get("name") == f"refs/heads/{branch}"), None)


def ado_create_branch(gp, branch, commit):
    res = ado(gp, "POST", "/refs", body=[{"name": f"refs/heads/{branch}", "oldObjectId": "0" * 40,
                                          "newObjectId": commit}]).get("value", [{}])
    if not res or not res[0].get("success"):
        raise HttpError(None, (res[0] if res else {}).get("updateStatus", "Failed"),
                        f"couldn't create branch {branch}", branch)


def ado_lock(gp, branch, locked=True):
    ado(gp, "PATCH", "/refs", params={"filter": f"heads/{branch}"}, body={"isLocked": locked})


def ado_delete_branch(gp, branch, commit):
    res = ado(gp, "POST", "/refs", body=[{"name": f"refs/heads/{branch}", "oldObjectId": commit,
                                          "newObjectId": "0" * 40}]).get("value", [{}])
    if not res or not res[0].get("success"):
        raise HttpError(None, (res[0] if res else {}).get("updateStatus", "Failed"), f"couldn't delete {branch}", branch)


def ado_default_head(gp):
    """(default branch name, its head commit) of the repository."""
    repo = ado(gp, "GET", "")
    name = (repo.get("defaultBranch") or "refs/heads/main")[len("refs/heads/"):]
    ref = ado_ref(gp, name)
    return name, (ref or {}).get("objectId")


def ado_tree(gp, commit, missing_ok=False):
    """Every file path under the connection's Git folder at a commit ([] if the folder is missing and missing_ok)."""
    scope = "/" + (gp.get("directoryName") or "").strip("/")
    try:
        body = ado(gp, "GET", "/items", params={"scopePath": scope, "recursionLevel": "Full",
                                                "versionDescriptor.version": commit,
                                                "versionDescriptor.versionType": "commit"})
    except HttpError as exc:
        if missing_ok and exc.status == 404:
            return []
        raise
    return [e["path"] for e in body.get("value", []) if e.get("gitObjectType") == "blob"]


def ado_file(gp, commit, path):
    return ado(gp, "GET", "/items", params={"path": path, "versionDescriptor.version": commit,
                                            "versionDescriptor.versionType": "commit", "$format": "octetStream"},
               raw=True)


def git_items_at(gp, commit):
    """Fabric items saved in Git at a commit: {key_of(type, name): {type, displayName, logicalId, folder, files}}."""
    paths = ado_tree(gp, commit)
    out = {}
    for p in paths:
        if not p.endswith("/.platform"):
            continue
        folder = p[: -len("/.platform")]
        try:
            meta = json.loads(ado_file(gp, commit, p).decode("utf-8-sig"))
        except ValueError as exc:
            raise HttpError(None, "BadPlatformFile", f"{p} is not valid JSON: {exc}", p) from None
        md, cfg = meta.get("metadata") or {}, meta.get("config") or {}
        files = [q for q in paths if q.startswith(folder + "/")]
        out[key_of(md.get("type"), md.get("displayName"))] = {
            "type": md.get("type"), "displayName": md.get("displayName"), "logicalId": cfg.get("logicalId"),
            "folder": folder, "files": len(files)}
    return out


# ---------------------------------------------------------------- Fabric Git integration
FOLDER_MARKER = "README.md"     # placeholder V2_1 writes so the workspace's Git folder exists before linking


def folder_marker_path(folder):
    return "/" + folder.strip("/") + "/" + FOLDER_MARKER


def ado_create_folder(gp, branch, folder, text):
    """Create a Git folder by pushing a small marker file to the branch: the Fabric Git connect API (unlike the
    workspace settings screen) doesn't create a missing folder and fails with GitProviderResourceNotFound.
    Returns the new branch head."""
    ref = ado_ref(gp, branch)
    if not ref:
        raise HttpError(None, "BranchNotFound", f"branch {branch} doesn't exist", branch)
    res = ado(gp, "POST", "/pushes", body={
        "refUpdates": [{"name": f"refs/heads/{branch}", "oldObjectId": ref["objectId"]}],
        "commits": [{"comment": f"Create Git folder {folder} for Fabric Git integration",
                     "changes": [{"changeType": "add", "item": {"path": folder_marker_path(folder)},
                                  "newContent": {"content": text, "contentType": "rawtext"}}]}]})
    return ((res.get("commits") or [{}])[0]).get("commitId")


def ado_tree_ids(gp, commit):
    """{path: blob objectId} under the connection's Git folder at a commit: compares folder content cheaply."""
    scope = "/" + (gp.get("directoryName") or "").strip("/")
    body = ado(gp, "GET", "/items", params={"scopePath": scope, "recursionLevel": "Full",
                                            "versionDescriptor.version": commit,
                                            "versionDescriptor.versionType": "commit"})
    return {e["path"]: e.get("objectId") for e in body.get("value", []) if e.get("gitObjectType") == "blob"}


def ado_create_tag(gp, name, commit, message):
    """Annotated tag (refs/tags/<name>) on a commit. Returns the tag's commit, or raises."""
    existing = ado(gp, "GET", "/refs", params={"filter": f"tags/{name}"}).get("value", [])
    if any(r.get("name") == f"refs/tags/{name}" for r in existing):
        raise HttpError(None, "TagExists", f"tag {name} already exists", name)
    return ado(gp, "POST", "/annotatedtags", body={"name": name, "taggedObject": {"objectId": commit},
                                                    "message": message})


# ---------------------------------------------------------------- who runs this, and their workspace role
def token_claims(audience="pbi"):
    """Claims of the runner's Entra token (no secret is printed): oid, upn, idtyp ('app' = service principal)."""
    try:
        part = notebookutils.credentials.getToken(audience).split(".")[1]
        return json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))
    except Exception:
        return {}


def me():
    c = token_claims()
    return {"oid": str(c.get("oid", "")).lower(), "upn": c.get("upn") or c.get("unique_name") or c.get("appid", ""),
            "tid": str(c.get("tid", "")).lower(),
            "isApp": c.get("idtyp") == "app" or ("appid" in c and "upn" not in c and "unique_name" not in c)}


def _tenant_from_headers(headers):
    tid = headers.get("X-VSS-ResourceTenant") or headers.get("x-vss-resourcetenant")
    if tid is None:
        m = re.search(r"login\.(?:microsoftonline\.com|windows\.net)/([0-9a-fA-F-]{36})",
                      headers.get("WWW-Authenticate") or headers.get("www-authenticate") or "")
        tid = m.group(1) if m else None
    if tid is None:
        return None
    tid = tid.split(",")[0].strip().lower()
    return "" if tid in ("", "00000000-0000-0000-0000-000000000000") else tid


def sp_settings_missing():
    return [n for n in ("ADO_SP_TENANT_ID", "ADO_SP_CLIENT_ID", "ADO_SP_SECRET_NAME")
            if not str(globals().get(n) or "").strip() or "<" in str(globals().get(n))]


def sp_repo_check(org, project, repo):
    """(True, '') if the service principal in ADO_SP_* can read the repository, else (False, error)."""
    url = f"https://dev.azure.com/{quote(org)}/{quote(project)}/_apis/git/repositories/{quote(repo)}"
    try:
        ado_call(url, auth=sp_header())
        return True, ""
    except HttpError as exc:
        return False, exc.short()


def ado_org_tenant_info(org, project=None, repo=None):
    """(tenant ID, how it was found) of the Entra tenant the DevOps organization is connected to. '' = not connected
    to Entra, None = couldn't tell. 1) Azure DevOps names it in the X-VSS-ResourceTenant (or WWW-Authenticate) header
    of an anonymous call. 2) Otherwise, if the service principal in ADO_SP_* can read the repository, the organization
    is connected to the service principal's tenant: Azure DevOps only accepts service principals of its own tenant."""
    for path in ("_apis/projects?api-version=" + ADO_API_VERSION, "", "_apis/connectionData"):
        try:
            resp = _session().get(f"https://dev.azure.com/{quote(org)}/{path}", timeout=30, allow_redirects=False)
        except requests.RequestException:
            continue
        tid = _tenant_from_headers(resp.headers)
        if tid is not None:
            return tid, "named by Azure DevOps"
    if project and repo and not sp_settings_missing():
        try:
            ok, _ = sp_repo_check(org, project, repo)
        except HttpError:
            ok = False
        if ok:
            return (str(ADO_SP_TENANT_ID).strip().lower(),
                    f"the organization accepts service principal {str(ADO_SP_CLIENT_ID).strip()} of tenant "
                    f"{str(ADO_SP_TENANT_ID).strip()}, and Azure DevOps only accepts service principals of the Entra "
                    "tenant it's connected to")
    return None, ("Azure DevOps didn't name its tenant, and the service principal couldn't confirm it" if project
                  else "Azure DevOps didn't name its tenant")


def ado_org_tenant(org, project=None, repo=None):
    return ado_org_tenant_info(org, project, repo)[0]


def my_workspace_role(ws_id):
    """Your role in a workspace via the admin API (Fabric administrator): 'Admin', 'Member', ..., '' = none,
    None = couldn't tell."""
    who = me()
    try:
        users = fabric("GET", f"/admin/workspaces/{ws_id}/users").get("accessDetails") or []
    except HttpError:
        return None
    for u in users:
        pr = u.get("principal") or {}
        upn = ((pr.get("userDetails") or {}).get("userPrincipalName") or "").lower()
        if str(pr.get("id", "")).lower() == who["oid"] or (upn and upn == str(who["upn"]).lower()):
            return (u.get("workspaceAccessDetails") or {}).get("workspaceRole") or ""
    return ""


def add_me_as_workspace_admin(ws_id):
    """Fabric/Power BI administrator API: add the runner as Admin of a workspace."""
    who = me()
    http("POST", f"https://api.powerbi.com/v1.0/myorg/admin/groups/{ws_id}/users", "pbi",
         json_body={"identifier": who["oid"], "principalType": "App" if who["isApp"] else "User",
                    "groupUserAccessRight": "Admin"}, label="fabric")


CROSS_GEO_TITLE = "Users can export items to Git repositories in other geographical locations"
CROSS_GEO_FIX = ("The workspace's capacity is in another geography than the Azure DevOps organization. A Fabric "
                 "administrator must enable Admin portal > Tenant settings > Git integration > '" + CROSS_GEO_TITLE +
                 "' for the entire organization, or for a security group that includes everyone who runs V2_1 to "
                 "V2_5 (the setting applies to the person doing the Git operation, not to the DevOps account). Only "
                 "item definitions cross geographies, never data. Wait a few minutes after Apply, then rerun: the "
                 "notebook resumes. Linking by hand fails the same way until the setting is on.")


def git_tenant_settings():
    """Git integration tenant settings from the admin API, by title (admin role needed)."""
    res = fabric("GET", "/admin/tenantsettings")
    return {t.get("title", t.get("settingName", "")): t for t in res.get("tenantSettings", [])
            if "git" in (t.get("title", "") + t.get("tenantSettingGroup", "")).lower()}


def git_connection_url(org, project, repo):
    return f"https://dev.azure.com/{quote(org)}/{quote(project)}/_git/{quote(repo)}/"


def find_git_connection(name):
    """Fabric connection with this display name that you can use, or None."""
    return next((c for c in fabric_pages("/connections") if c.get("displayName") == name), None)


def create_git_connection(name, org, project, repo):
    """Fabric cloud connection 'Azure DevOps - Source control' with the service principal in ADO_SP_*. The
    client secret is read from Key Vault and only sent to Fabric."""
    secret = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_SP_SECRET_NAME))
    body = {"displayName": name, "connectivityType": "ShareableCloud",
            "connectionDetails": {"type": "AzureDevOpsSourceControl", "creationMethod": "AzureDevOpsSourceControl.Contents",
                                  "parameters": [{"dataType": "Text", "name": "url",
                                                  "value": git_connection_url(org, project, repo)}]},
            "credentialDetails": {"credentials": {"credentialType": "ServicePrincipal",
                                                  "tenantId": str(ADO_SP_TENANT_ID).strip(),
                                                  "servicePrincipalClientId": str(ADO_SP_CLIENT_ID).strip(),
                                                  "servicePrincipalSecret": secret}}}
    return fabric("POST", "/connections", body)


GIT_LINK_MODES = {"service_principal": "service principal (Fabric connection)",
                  "my_account": "your own Azure DevOps sign-in",
                  "connection_id": "an existing Fabric connection (GIT_CONNECTION_ID)"}


def git_link_mode():
    """How Fabric signs in to Azure DevOps for the workspaces' Git link (GIT_LINK_AUTH)."""
    raw = globals().get("GIT_LINK_AUTH")
    if raw is None:                       # older settings cells
        if globals().get("CREATE_GIT_CONNECTION"):
            return "service_principal"
        return "connection_id" if str(globals().get("GIT_CONNECTION_ID") or "").strip() else "my_account"
    m = str(raw).strip().lower().replace("-", "_").replace(" ", "_")
    m = {"sp": "service_principal", "spn": "service_principal", "serviceprincipal": "service_principal",
         "user": "my_account", "me": "my_account", "automatic": "my_account", "sso": "my_account",
         "connection": "connection_id", "configuredconnection": "connection_id"}.get(m, m)
    if m not in GIT_LINK_MODES:
        raise ValueError(f"GIT_LINK_AUTH = {raw!r}: use \"service_principal\", \"my_account\" or \"connection_id\"")
    return m


def _read_git_connection(cid):
    try:
        c = fabric("GET", f"/connections/{cid}")
    except HttpError as exc:
        raise RuntimeError(f"Can't read Fabric connection {cid} ({exc.short()}): check GIT_CONNECTION_ID, and that you "
                           "have User permission on it (Settings > Manage connections and gateways).") from None
    t = (c.get("connectionDetails") or {}).get("type")
    if t != "AzureDevOpsSourceControl":
        raise RuntimeError(f"Fabric connection {cid} ({c.get('displayName')}) is of type {t}, not Azure DevOps - "
                           "Source control: choose another connection.")
    return c


def plan_git_link(who, org, project, repo, allow_create=True):
    """How Fabric will sign in to Azure DevOps when it links a workspace, from GIT_LINK_AUTH. Returns {mode,
    connectionId, create (name of the connection to create on confirm, or None), text}. Raises before anything
    changes when the choice can't work."""
    mode = git_link_mode()
    cid = str(globals().get("GIT_CONNECTION_ID") or "").strip()
    if mode == "my_account":
        if cid:
            raise ValueError("GIT_CONNECTION_ID is filled but GIT_LINK_AUTH = \"my_account\": empty GIT_CONNECTION_ID, "
                             "or set GIT_LINK_AUTH = \"connection_id\" to use that connection.")
        if who["isApp"]:
            raise RuntimeError("This notebook runs as a service principal / workspace identity, which has no Azure "
                               "DevOps sign-in of its own: set GIT_LINK_AUTH = \"service_principal\".")
        return {"mode": mode, "connectionId": "", "create": None,
                "text": f"your own Azure DevOps sign-in ({who['upn']}). That only works if the DevOps organization "
                        f"is connected to the same Entra tenant as Fabric and {who['upn']} is a member of it, with "
                        "Basic access and project Contributors."}
    if mode == "connection_id":
        if not cid:
            raise ValueError("GIT_LINK_AUTH = \"connection_id\" needs GIT_CONNECTION_ID.")
        c = _read_git_connection(cid)
        return {"mode": mode, "connectionId": cid, "create": None,
                "text": f"the identity stored in Fabric connection {c.get('displayName')} ({cid}). Give that identity "
                        "Basic access and project Contributors in Azure DevOps."}
    need = sp_settings_missing()
    if need:
        raise ValueError("GIT_LINK_AUTH = \"service_principal\" needs: " + ", ".join(need) +
                         " (fill them even if ADO_AUTH = \"pat\").")
    sp_tid = str(ADO_SP_TENANT_ID).strip().lower()
    if who.get("tid") and sp_tid != who["tid"] and not globals().get("ALLOW_CROSS_TENANT"):
        raise RuntimeError(f"Stopped: the service principal's tenant ({sp_tid}) isn't the Fabric tenant ({who['tid']}). "
                           "Use an app registration from the same tenant (ALLOW_CROSS_TENANT is not part of this "
                           "migration).")
    sp_text = (f"service principal {str(ADO_SP_CLIENT_ID).strip()} (not tied to a person). Fabric stores its client "
               "secret in the connection: when you renew the secret, update the connection too.")
    if cid:
        c = _read_git_connection(cid)
        return {"mode": mode, "connectionId": cid, "create": None,
                "text": f"Fabric connection {c.get('displayName')} ({cid}), which signs in as the " + sp_text}
    name = str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{org}-{project}-{repo}"
    found = find_git_connection(name)
    if found:
        t = (found.get("connectionDetails") or {}).get("type")
        cred = (found.get("credentialDetails") or {}).get("credentialType")
        if t != "AzureDevOpsSourceControl" or (cred and cred != "ServicePrincipal"):
            raise RuntimeError(f"A Fabric connection named {name} already exists but it's {t} / {cred}, not Azure DevOps "
                               "- Source control with a service principal: set another GIT_CONNECTION_NAME.")
        return {"mode": mode, "connectionId": found["id"], "create": None,
                "text": f"existing Fabric connection {name} ({found['id']}), which signs in as the " + sp_text}
    if not allow_create:
        return {"mode": mode, "connectionId": "", "create": name,
                "text": f"Fabric connection {name} (doesn't exist yet), which will sign in as the " + sp_text}
    return {"mode": mode, "connectionId": "", "create": name,
            "text": f"Fabric connection {name}, created when you confirm the plan, which signs in as the " + sp_text}


SAME_TENANT_FIX = (
    "This migration keeps everything in one Entra tenant: the Azure DevOps organization must be connected to the "
    "same Entra tenant as Fabric. Either use a DevOps organization that is already in this tenant, or connect this "
    "one: Azure DevOps > Organization settings > Microsoft Entra > Connect directory (Project Collection "
    "Administrator; existing users must exist in the tenant or they lose access, so plan it with the DevOps admin). "
    "Then add the people who run the notebooks (Basic access, project Contributors), open a workspace > Source "
    "control once and sign in. Linking across tenants (ALLOW_CROSS_TENANT = True) is not part of this migration.")
FOLDER_FIRST = (
    "Most common cause: the Git folder doesn't exist in the branch (the connect API doesn't create it, unlike the "
    "workspace settings screen). V2_1 creates it before linking; if you link by hand or with another tool, create "
    "the folder first (a file inside it, for example README.md). ")
GIT_NOT_FOUND_FIX = (
    "Fabric couldn't see the organization, project, repository or branch with your own Azure DevOps sign-in. This "
    "notebook already found all of them with the PAT/service principal, so the names are right (those settings are "
    "only for the notebook's own DevOps calls). With your own sign-in Fabric only sees DevOps organizations connected "
    "to this Fabric tenant's Entra ID in which you are a member. Check, in this order: (1) you are a member of the "
    "organization with Basic access and in the project's Contributors; (2) the organization is connected to this "
    "Entra tenant (V2_0 row 'organization in the same Entra tenant as Fabric'). " + SAME_TENANT_FIX + " Optional, "
    "still within the same tenant: GIT_LINK_AUTH = \"service_principal\" links with the service principal "
    "instead of your own account.")
GIT_NOT_FOUND_WITH_CONNECTION_FIX = (
    "Fabric couldn't see the repository, branch or folder with the Git connection's identity. " + FOLDER_FIRST +
    "Otherwise: Fabric couldn't see the repository with the Git connection's identity. Give that identity (for example the "
    "service principal) access: Azure DevOps > Organization settings > Users > Add (Basic), and the project's "
    "Contributors group. Check that the connection URL points to this organization/project/repository, and that "
    "the connection's tenant ID is the tenant the DevOps organization is connected to.")


def git_connect_fix(exc):
    """How to fix a failed Git connect / initialize call."""
    text = f"{getattr(exc, 'code', '')} {exc}".lower()
    if "principaltypenotsupported" in text or "service principal" in text:
        return ("The notebook runs as a service principal / workspace identity: Fabric only accepts that with a "
                "stored connection: set GIT_LINK_AUTH = \"service_principal\", or run the notebook interactively as "
                "a user.")
    if "gitproviderresourcenotfound" in text:
        return FOLDER_FIRST + "Otherwise: " + GIT_NOT_FOUND_FIX if git_link_mode() == "my_account" else \
            GIT_NOT_FOUND_WITH_CONNECTION_FIX
    if "insufficientprivileges" in text or getattr(exc, "status", None) == 403:
        return ("You must be Admin of the workspace (Fabric administrator isn't enough). Set "
                "ADD_ME_AS_WORKSPACE_ADMIN = True, or ask a workspace admin to add you.")
    if "credential" in text or "gitprovider" in text or "tf400813" in text or "unauthorized" in text:
        return ("Fabric has no working Azure DevOps credentials for you. Either open any workspace > Source control "
                "once and sign in to Azure DevOps (same Entra tenant as the DevOps organization), or set "
                "GIT_LINK_AUTH = \"service_principal\" so Fabric signs in with the service principal instead.")
    if "crossgeo" in text or "cross geo" in text or "cross-geo" in text:
        return CROSS_GEO_FIX
    if "capacity" in text:
        return "Assign the workspace to a Fabric capacity first."
    if "alreadyconnected" in text:
        return "Already connected: rerun the notebook, it will check the existing link."
    if "directory" in text or "folder" in text or "branch" in text:
        return "Check the branch and folder: the folder must be empty or belong to this workspace."
    return "See the error; if it persists, link this workspace by hand with the values in the table."


def current_workspace_id():
    """Workspace this notebook runs in ('' if unknown)."""
    try:
        ctx = notebookutils.runtime.context
        return str(ctx.get("currentWorkspaceId") or "").lower()
    except Exception:
        return ""


RUN_FROM_ELSEWHERE = ("Run the migration notebooks from a separate workspace that isn't migrated (for example "
                      "'fabric-migration-control'): this workspace is in scope, so its notebooks, including the one "
                      "running, would be deleted with the other items.")


def my_git_credentials(ws_id):
    """Your own Git credentials for a linked workspace ({'source': 'Automatic' | 'ConfiguredConnection' | 'None',
    'connectionId'}). Per user and per workspace: the workspace screens (Source control, Git integration) use them."""
    try:
        return fabric("GET", f"/workspaces/{ws_id}/git/myGitCredentials")
    except HttpError as exc:
        return {"source": "?", "error": exc.short()}


def creds_need_fix(cred, connection_id):
    """True if your Git credentials for the workspace should be switched to the Fabric connection."""
    if git_link_mode() == "my_account":
        return False
    return cred.get("source") != "ConfiguredConnection" or bool(connection_id and cred.get("connectionId") != connection_id)


def set_my_git_credentials(ws_id, connection_id):
    fabric("PATCH", f"/workspaces/{ws_id}/git/myGitCredentials",
           {"source": "ConfiguredConnection", "connectionId": connection_id})


CREDS_WHY = ("your own Git sign-in for this workspace is '{src}', so the workspace screens (Source control, Git "
             "integration) try your personal Azure DevOps account and show 'Can't fetch branches' / 'connection "
             "removed'. The link itself is fine.")


def git_connection_label():
    """Name of the Fabric connection used to link (GIT_CONNECTION_NAME or the default name)."""
    if git_link_mode() == "connection_id":
        return f"with ID {globals().get('GIT_CONNECTION_ID')}"
    return (str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{globals().get('ADO_ORGANIZATION')}"
            f"-{globals().get('ADO_PROJECT')}-{globals().get('ADO_REPOSITORY')}")


def git_creds_hint(ws_id, gp=None):
    """Plain-language fix for a Git call (status, commit, update, disconnect) that fails because of YOUR OWN Git
    sign-in for the workspace. The notebooks don't change that sign-in by themselves: you pick it in the workspace."""
    cred = my_git_credentials(ws_id)
    gp = gp or {}
    org, project, repo = (gp.get("organizationName") or globals().get("ADO_ORGANIZATION", ""),
                          gp.get("projectName") or globals().get("ADO_PROJECT", "<project>"),
                          gp.get("repositoryName") or globals().get("ADO_REPOSITORY", "<repository>"))
    name = str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{org}-{project}-{repo}"
    src = cred.get("source")
    now = f"Your Git sign-in for this workspace is '{src}'" + (
        f" (connection {cred.get('connectionId')})" if cred.get("connectionId") else "") + "."
    return (f"{now} Git calls run with YOUR OWN Git sign-in for each workspace. Open the workspace > Source control "
            f"> account/credentials and choose the Fabric connection '{name}' (the one used to link it; if it isn't "
            "listed, ask its owner to share it with you: Manage connections and gateways > Manage users), then rerun. "
            "If the workspace was linked with your own account (GIT_LINK_AUTH = \"my_account\"), sign in to Azure "
            "DevOps there instead.")


def git_fail_text(exc, ws_id, gp=None):
    """Full error text for a failed Fabric Git call plus the fix: role (403) or your own Git sign-in (other errors)."""
    text = exc.short()
    if "/git/" not in str(getattr(exc, "url", "") or ""):
        return text
    if exc.status == 403:
        return (text + " | FIX: Git calls need Contributor (status, commit, update) or Admin (connect, disconnect) on "
                "the workspace for the account running the notebook.")
    try:
        return text + " | FIX: " + git_creds_hint(ws_id, gp)
    except Exception:          # the hint is best effort; never hide the original error
        return text


def git_connection(ws_id):
    return fabric("GET", f"/workspaces/{ws_id}/git/connection")


def git_status(ws_id):
    return fabric("GET", f"/workspaces/{ws_id}/git/status")


def resolve_target_workspace(old_id, name, override=None, recorded=None):
    """(workspace ID after the move, how it was found) or (None, why not). A home-region move keeps the workspace
    IDs, so the same ID is tried first; if it's gone, the workspace with the same name (names are unique in a
    tenant). override = WORKSPACE_MAP entry; recorded = the target V2_5 already used (restore_state.json)."""
    if override:
        return override, "WORKSPACE_MAP"
    if recorded:
        return recorded, "same as V2_5" + ("" if recorded == old_id else " (ID changed)")
    try:
        fabric("GET", f"/workspaces/{old_id}")
        return old_id, "same ID"
    except HttpError as exc:
        if exc.status not in (403, 404):
            raise
    try:
        found = [w for w in fabric_pages("/workspaces") if (w.get("displayName") or "") == name]
    except HttpError as exc:
        return None, f"workspace {old_id} not found and the workspace list can't be read ({exc.short()})"
    if len(found) == 1:
        return norm_id(found[0]["id"]), f"found by name; ID changed from {old_id}"
    if not found:
        return None, (f"workspace {old_id} ({name}) not found, and no workspace you can access has that name. "
                      "Recreate it or add yourself to it; if it was renamed, set WORKSPACE_MAP")
    return None, f"workspace {old_id} not found and {len(found)} workspaces are named {name}: set WORKSPACE_MAP"


def workspace_items(ws_id):
    return [{"id": i["id"], "type": i["type"], "displayName": i["displayName"], "workspaceId": ws_id}
            for i in fabric_pages(f"/workspaces/{ws_id}/items")]


# ---------------------------------------------------------------- migration records (ADLS)
def migration_prefix(snapshot_id):
    return f"{ADLS_ROOT.strip('/')}/_migration/{snapshot_id}"


def plan_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]


def load_manifest(snapshot_id):
    """The manifest written by the snapshot notebook, checked against its stored hash."""
    if not snapshot_id or "<" in snapshot_id:
        raise ValueError("Set SNAPSHOT_ID to the snapshot ID printed by the snapshot notebook.")
    prefix = migration_prefix(snapshot_id)
    try:
        raw = adls_read(f"{prefix}/manifest.json")
        stored = adls_read(f"{prefix}/manifest.sha256").decode().strip()
    except HttpError as exc:
        raise RuntimeError(f"Can't read the snapshot manifest under {prefix}/ ({exc.short()}).") from None
    if hashlib.sha256(raw).hexdigest() != stored:
        raise RuntimeError("The manifest doesn't match its stored hash: it was changed after the snapshot. Stop and "
                           "investigate; take a new snapshot.")
    return json.loads(raw)


def save_log(prefix, name):
    df = pd.DataFrame(RESULTS, columns=["Workspace", "Item", "Type", "ItemId", "Step", "Status", "Detail"])
    adls_write(f"{prefix}/{name}_{NOW_ID}.csv", df.to_csv(index=False))
    return df

## 1. Workspaces and items
Lists every item in each workspace and sorts it: in-scope Fabric items, other Fabric items (deleted in the migration,
must be acknowledged) and Power BI items (they move with the workspace and are left alone).

In [ ]:
ADO_ORGANIZATION = ado_org_name(ADO_ORGANIZATION)   # a URL works too
SNAPSHOT_ID = NOW_ID
missing = [n for n in ("ADO_ORGANIZATION", "KEY_VAULT_URL", "ADLS_ACCOUNT",
                       "SECONDARY_BACKUP_RUN_ID") if "<" in str(globals()[n])] + ado_auth_missing()
if missing:
    raise ValueError("Fill in these settings first: " + ", ".join(missing))
ws_ids = [norm_id(w) for w in WORKSPACE_IDS]
if not ws_ids or None in ws_ids:
    raise ValueError("WORKSPACE_IDS must list at least one workspace ID (GUID).")
NO_GIT = {norm_id(w) for w in NO_GIT_WORKSPACE_IDS}
if None in NO_GIT or NO_GIT - set(ws_ids):
    raise ValueError("NO_GIT_WORKSPACE_IDS must be workspace IDs (GUIDs) that are also in WORKSPACE_IDS.")
if NO_GIT and not SAVE_RAW_COPY_TO_ADLS:
    raise ValueError("Workspaces without Git (NO_GIT_WORKSPACE_IDS) are restored from the raw copy: set "
                     "SAVE_RAW_COPY_TO_ADLS = True.")
if ALLOW_SECONDARY_ONLY_ITEMS and not SAVE_RAW_COPY_TO_ADLS:
    raise ValueError("ALLOW_SECONDARY_ONLY_ITEMS = True needs SAVE_RAW_COPY_TO_ADLS = True: V2_4 compares those items "
                     "with their raw copy to prove they didn't change.")

WS = {}
for ws_id in dict.fromkeys(ws_ids):
    try:
        info = fabric("GET", f"/workspaces/{ws_id}")
        items = workspace_items(ws_id)
    except HttpError as exc:
        raise RuntimeError(f"Can't read workspace {ws_id} ({exc.short()}). You need Contributor or higher on it.") \
            from None
    name = info.get("displayName", ws_id)
    for i in items:
        i["workspaceName"] = name
    ws = {"id": ws_id, "name": name, "capacityId": info.get("capacityId"), "blockers": [], "git": {},
          "noGit": ws_id in NO_GIT,
          "items": [i for i in items if i["type"] not in POWER_BI_TYPES],
          "pbi": [i for i in items if i["type"] in POWER_BI_TYPES]}
    for i in ws["items"]:
        i["staging"] = is_dataflow_staging(i)
        if i["type"] == "Lakehouse" and not i["staging"]:
            try:
                props = fabric("GET", f"/workspaces/{ws_id}/lakehouses/{i['id']}").get("properties") or {}
                i["sqlEndpointId"] = (props.get("sqlEndpointProperties") or {}).get("id") or ""
                i["defaultSchema"] = props.get("defaultSchema") or ""      # set only on schema-enabled Lakehouses
            except HttpError as exc:
                log(name, i, "Inventory", "Warning", f"can't read the SQL endpoint ID: {exc.short()}")
        if i["type"] == "KQLDatabase":
            try:
                props = fabric("GET", f"/workspaces/{ws_id}/kqlDatabases/{i['id']}").get("properties") or {}
                i["parentEventhouseItemId"] = props.get("parentEventhouseItemId") or ""
            except HttpError as exc:
                log(name, i, "Inventory", "Warning", f"can't read the parent Eventhouse: {exc.short()}")
    WS[ws_id] = ws
    other = sorted({i["type"] for i in ws["items"] if i["type"] not in ITEM_TYPES})
    print(f"{name}: {len(ws['items'])} Fabric items, {len(ws['pbi'])} Power BI items"
          + (f"; other Fabric types: {', '.join(other)}" if other else ""))


def block(ws, step, reason):
    ws["blockers"].append(f"{step}: {reason}")
    log(ws["name"], {}, step, "Blocked", reason)

## 2. Git sync check (every workspace)
Read-only: for each workspace, the Git connection, Source control changes and updates, and the DevOps branch head.
**Synced** = connected to Azure DevOps, nothing uncommitted, nothing incoming, no conflicts. The branch is shared,
so a branch head that moved only through other workspaces' folders still counts as synced.
With `REQUIRE_ALL_WORKSPACES_SYNCED = True` (default) the snapshot is taken only when every workspace is synced.
This notebook doesn't commit or update for you unless `COMMIT_PENDING_CHANGES = True`; the Fix column says what to
do in the workspace's Source control.

In [ ]:
# Sync check: read-only for every workspace first. Nothing is committed or updated by this cell unless
# COMMIT_PENDING_CHANGES = True, DRY_RUN = False and (with REQUIRE_ALL_WORKSPACES_SYNCED) nothing else is out of sync.
SYNC_ROWS = []
SYNC_OK = ("Synced", "Uncommitted (will be committed)", "No Git (ADLS route)")


def sync_row(ws, status, detail="", fix="", **extra):
    g = ws.get("git") or {}
    ws["sync"] = status
    SYNC_ROWS.append({"Workspace": ws["name"], "Sync": status, "Branch": g.get("branchName") or "",
                      "Git folder": g.get("directoryName") or "", "Workspace commit": extra.get("wh") or "",
                      "DevOps branch head": extra.get("head") or "", "Uncommitted": extra.get("out", 0),
                      "Incoming": extra.get("inc", 0), "Detail": detail, "Fix": fix})
    if status not in SYNC_OK:
        block(ws, "Git sync", f"{status}: {detail}" + (f" | FIX: {fix}" if fix else ""))


def check_sync(ws):
    name, ws_id = ws["name"], ws["id"]
    if ws.get("noGit"):
        try:
            state = git_connection(ws_id).get("gitConnectionState")
        except HttpError:
            state = None
        return sync_row(ws, "No Git (ADLS route)", "in NO_GIT_WORKSPACE_IDS: Git isn't used; V2_5 recreates the items "
                        "from the raw copy in ADLS" + (f" (the workspace is {state}; that link is ignored)"
                                                       if state not in (None, "NotConnected") else ""))
    try:
        conn = git_connection(ws_id)
    except HttpError as exc:
        return sync_row(ws, "Can't read", f"Git connection: {git_fail_text(exc, ws_id)}",
                        "You need Contributor or higher on the workspace.")
    gp = conn.get("gitProviderDetails") or {}
    ws["git"] = {k: gp.get(k) for k in ("gitProviderType", "organizationName", "projectName", "repositoryName",
                                         "branchName", "directoryName")}
    state = conn.get("gitConnectionState")
    if state != "ConnectedAndInitialized":
        return sync_row(ws, "Not connected", f"Git connection state: {state}",
                        "Connect it with V2_1_Connect_Workspaces_To_Git (or Workspace settings > Git integration).")
    if gp.get("gitProviderType") != "AzureDevOps":
        return sync_row(ws, "Wrong provider", f"connected to {gp.get('gitProviderType')}, not Azure DevOps",
                        "Reconnect it to the Azure DevOps repository (V2_1).")
    if str(gp.get("organizationName", "")).lower() != ADO_ORGANIZATION.lower():
        return sync_row(ws, "Wrong organization", f"connected to {gp.get('organizationName')}, not {ADO_ORGANIZATION}",
                        f"Reconnect it to {ADO_ORGANIZATION} (V2_1), or fix ADO_ORGANIZATION.")
    try:
        st = git_status(ws_id)
    except HttpError as exc:
        return sync_row(ws, "Git status failed", git_fail_text(exc, ws_id, gp))
    wh, remote = st.get("workspaceHead"), st.get("remoteCommitHash")
    try:
        head = (ado_ref(gp, gp.get("branchName")) or {}).get("objectId")
    except HttpError as exc:
        return sync_row(ws, "Can't read DevOps", f"branch {gp.get('branchName')}: {exc.short()}", wh=wh,
                        fix="Check the DevOps sign-in of this notebook (ADO_AUTH) and its Code (Read) access.")
    if not head:
        return sync_row(ws, "Branch missing", f"branch {gp.get('branchName')} isn't in the repository", wh=wh,
                        fix="Reconnect the workspace to an existing branch (V2_1).")
    changes = st.get("changes") or []
    incoming = [c for c in changes if c.get("remoteChange") or c.get("conflictType") not in (None, "None")]
    outgoing = [c for c in changes if c.get("workspaceChange") and c not in incoming]
    deletions = [c for c in outgoing if c["workspaceChange"] == "Deleted"]
    for c in changes:
        md = c.get("itemMetadata") or {}
        log(name, {"type": md.get("itemType"), "displayName": md.get("displayName")}, "Git status", "Info",
            f"workspace: {c.get('workspaceChange') or '-'}, Git: {c.get('remoteChange') or '-'}, "
            f"conflict: {c.get('conflictType') or 'None'}")
    ws["status_raw"], ws["outgoing"] = st, outgoing
    counts = {"wh": wh, "head": head, "out": len(outgoing), "inc": len(incoming)}
    if not wh:
        return sync_row(ws, "Never synced", "the workspace has no synced commit yet", **counts,
                        fix="Source control > Commit (or Update) once, then run this notebook again.")
    if incoming:
        return sync_row(ws, "Incoming changes", f"{len(incoming)} change(s) or conflict(s) in Git not in the workspace",
                        "Source control > Updates > Update all (resolve conflicts first), then run again.", **counts)
    if deletions and not ALLOW_COMMITTING_DELETIONS:
        names = ", ".join(f"{(c.get('itemMetadata') or {}).get('displayName')}" for c in deletions)
        return sync_row(ws, "Uncommitted deletions", f"{len(deletions)} item(s) in Git but deleted in the workspace "
                        f"({names}); committing would remove them from Git",
                        "Restore the items, or confirm they were deleted on purpose and set "
                        "ALLOW_COMMITTING_DELETIONS = True.", **counts)
    if wh != head:
        # The branch is shared: other workspaces' commits move its head. This workspace is still synced when its
        # own Git folder is the same at its commit and at the branch head.
        try:
            same = ado_tree_ids(gp, wh) == ado_tree_ids(gp, head)
        except HttpError as exc:
            return sync_row(ws, "Can't read DevOps", f"folder at {wh} / {head}: {exc.short()}", **counts)
        if not same:
            return sync_row(ws, "Behind the branch", f"its Git folder changed after commit {wh[:8]} (head {head[:8]})",
                            "Source control > Updates > Update all, then run again.", **counts)
    if outgoing:
        if not COMMIT_PENDING_CHANGES:
            return sync_row(ws, "Uncommitted changes", f"{len(outgoing)} workspace change(s) not committed to Git",
                            "Source control > Changes > Commit (or set COMMIT_PENDING_CHANGES = True), then run "
                            "again.", **counts)
        ws["pendingCommit"] = len(outgoing)
        ws["pendingAdded"] = {((c.get("itemMetadata") or {}).get("itemIdentifier") or {}).get("objectId")
                              for c in outgoing if c["workspaceChange"] == "Added"}
        ws["head"] = wh
        return sync_row(ws, "Uncommitted (will be committed)", f"{len(outgoing)} change(s); COMMIT_PENDING_CHANGES",
                        **counts)
    ws["head"] = wh
    sync_row(ws, "Synced", "" if wh == head else "branch moved on through other folders only", **counts)


for ws in WS.values():
    check_sync(ws)

NOT_SYNCED = [w["name"] for w in WS.values() if w.get("sync") not in SYNC_OK]
SYNC_STOP = REQUIRE_ALL_WORKSPACES_SYNCED and bool(NOT_SYNCED)
if SYNC_STOP:
    for ws in WS.values():
        if ws["name"] not in NOT_SYNCED:
            block(ws, "Git sync", f"not every workspace is synced ({', '.join(NOT_SYNCED)}); "
                  "REQUIRE_ALL_WORKSPACES_SYNCED takes the snapshot only when all are")

# Commit only with COMMIT_PENDING_CHANGES, a real run, and nothing else out of sync.
for ws in WS.values():
    if not ws.get("pendingCommit") or ws["blockers"]:
        continue
    if DRY_RUN:
        log(ws["name"], {}, "Git commit", "Dry run", f"would commit {ws['pendingCommit']} workspace change(s)")
        continue
    gp = ws["git"]
    try:
        fabric("POST", f"/workspaces/{ws['id']}/git/commitToGit",
               {"mode": "All", "workspaceHead": ws["status_raw"].get("workspaceHead"),
                "comment": f"Pre-migration snapshot {SNAPSHOT_ID}"}, want_result=False)
        st = git_status(ws["id"])
    except HttpError as exc:
        block(ws, "Git commit", f"commit failed ({git_fail_text(exc, ws['id'], gp)})")
        ws.pop("head", None)
        continue
    if st.get("changes"):
        block(ws, "Git commit", f"{len(st['changes'])} change(s) still pending after the commit (often an item type "
                                "that Git doesn't support). See the Git status rows.")
        ws.pop("head", None)
        continue
    ws["head"], ws["pendingCommit"] = st.get("workspaceHead") or st.get("remoteCommitHash"), 0
    log(ws["name"], {}, "Git commit", "OK", f"committed {len(ws['outgoing'])} change(s): {ws['head']}")

SYNC_DF = pd.DataFrame(SYNC_ROWS)
show(SYNC_DF)
synced = len(WS) - len(NOT_SYNCED)
print(f"Git sync: {synced} of {len(WS)} workspace(s) synced.")
if NOT_SYNCED:
    print(f"NOT SYNCED: {', '.join(NOT_SYNCED)}. Follow the Fix column, then run this notebook again."
          + (" Nothing will be created in this run (REQUIRE_ALL_WORKSPACES_SYNCED)." if SYNC_STOP else
             " Those workspaces are BLOCKED; the others continue."))
for ws in WS.values():
    if ws.get("head"):
        print(f"{ws['name']}: {ws['git']['repositoryName']} / {ws['git']['branchName']} / "
              f"{ws['git']['directoryName'] or '(root)'} at {ws['head']}"
              + (f" (+{ws['pendingCommit']} change(s) to commit)" if ws.get("pendingCommit") else ""))

## 3. What the snapshot commit contains
Reads the `.platform` file of every item folder in the commit and matches it to the workspace items by type and name.

In [ ]:
for ws in WS.values():
    if not ws.get("head"):
        continue
    try:
        saved = git_items_at(ws["git"], ws["head"])
    except HttpError as exc:
        block(ws, "Git content", f"can't read the repository at {ws['head']} ({exc.short()}). Check the Azure DevOps "
                                 "token in Key Vault and its Code (Read) scope.")
        continue
    ws["gitItems"] = saved
    for item in ws["items"] + ws["pbi"]:
        g = saved.get(key_of(item["type"], item["displayName"]))
        item["inGit"] = bool(g)
        item.update(logicalId=(g or {}).get("logicalId", ""), gitFolder=(g or {}).get("folder", ""),
                    gitFiles=(g or {}).get("files", 0))
        if not g and item["id"] in ws.get("pendingAdded", ()):
            item["inGit"] = "after commit"
    # Schema-enabled Lakehouse: Fabric recreates it from Git with schemas only if its definition says so
    # (lakehouse.metadata.json with defaultSchema). Without it the restored Lakehouse can't take the tables back.
    for item in ws["items"]:
        if item["type"] != "Lakehouse" or not item.get("defaultSchema") or item.get("inGit") is not True:
            continue
        try:
            meta = json.loads(ado_file(ws["git"], ws["head"], item["gitFolder"] + "/lakehouse.metadata.json")
                              .decode("utf-8-sig") or "{}")
        except (HttpError, ValueError):
            meta = {}
        if meta.get("defaultSchema"):
            log(ws["name"], item, "Lakehouse schemas", "OK", f"schema-enabled (default {meta['defaultSchema']}), "
                "kept in its Git definition")
        else:
            item["schemaProblem"] = ("schema-enabled Lakehouse, but its Git definition has no lakehouse.metadata.json "
                                     "with defaultSchema: restored from Git it would have no schemas and its tables "
                                     "couldn't be loaded back. Commit the Lakehouse again from Source control, then "
                                     "run again")
            log(ws["name"], item, "Lakehouse schemas", "Blocked", item["schemaProblem"])
    in_git = sum(1 for i in ws["items"] if i.get("inGit"))
    print(f"{ws['name']}: {len(saved)} item folders in Git; {in_git} of {len(ws['items'])} Fabric items found")

## 4. Secondary backup (ADLS)
Checks the V2_2 backup run: complete, started after the change freeze, an OK definition for every in-scope item
(files present in ADLS) and an OK data backup for every Lakehouse, Warehouse and KQL database.

In [ ]:
import io

RUN_PREFIX = f"{ADLS_ROOT.strip('/')}/{SECONDARY_BACKUP_RUN_ID}"
GOOD = ("OK", "Unchanged", "Not applicable")
run_problems = []
try:
    started = json.loads(adls_read(f"{RUN_PREFIX}/_run_started.json"))
    if not adls_exists(f"{RUN_PREFIX}/_run_completed.json"):
        run_problems.append(f"backup run {SECONDARY_BACKUP_RUN_ID} didn't finish (no _run_completed.json)")
    backup_log = pd.read_csv(io.BytesIO(adls_read(f"{RUN_PREFIX}/backup_log.csv")), dtype=str).fillna("")
except HttpError as exc:
    raise RuntimeError(f"Can't read backup run {RUN_PREFIX}/ ({exc.short()}). Check SECONDARY_BACKUP_RUN_ID and "
                       "your Storage Blob Data Reader/Contributor role.") from None
run_started = str(started.get("startedUtc", ""))
if not FREEZE_STARTED_UTC:
    run_problems.append("FREEZE_STARTED_UTC is empty: set when the change freeze started, so the check can prove the "
                        "backup is newer than the last change")
else:
    freeze = dt.datetime.fromisoformat(FREEZE_STARTED_UTC.replace("Z", "+00:00"))
    if not run_started or dt.datetime.fromisoformat(run_started.replace("Z", "+00:00")) < freeze:
        run_problems.append(f"backup run started {run_started or '(unknown)'}, before the freeze {FREEZE_STARTED_UTC}")
for p in run_problems:
    for ws in WS.values():
        block(ws, "Secondary backup", p)

STEP_STATUS = {(r.ItemId.lower(), r.Step): r.Status for r in backup_log.itertuples()}
for ws in WS.values():
    top = [n for n, _, is_dir in adls_list(RUN_PREFIX, recursive=False) if is_dir and n.endswith("__" + ws["id"])]
    files = adls_list(f"{RUN_PREFIX}/{top[0]}") if top else []
    def_files = Counter(p.split("/", 1)[0].rsplit("__", 1)[-1].lower() for p, _, d in files
                        if not d and "/definition/" in p)
    ws["secondaryFolder"] = f"{RUN_PREFIX}/{top[0]}" if top else ""
    for item in ws["items"]:
        if item["type"] not in ITEM_TYPES or item.get("staging"):
            continue
        steps = ["SQL scripts"] if item["type"] in CHILD_TYPES else ["Definition"]
        sec = {s: STEP_STATUS.get((item["id"].lower(), s), "") for s in steps}
        if item["type"] in DATA_TYPES:
            dstep, dstatus = data_step(item["type"], lambda s: STEP_STATUS.get((item["id"].lower(), s), ""))
            sec[dstep] = dstatus
        if "Definition" in sec and sec["Definition"] in GOOD and not def_files.get(item["id"].lower()):
            sec["Definition"] = "no files in ADLS"
        item["secondary"] = sec
    print(f"{ws['name']}: secondary backup folder {ws['secondaryFolder'] or 'NOT FOUND'}")

## 5. Readiness gate
One row per item with how it will be recovered and, if blocked, why. A workspace is READY only when every row is.

In [ ]:
rows = []
for ws in WS.values():
    for item in ws["items"]:
        t, reasons, recovery = item["type"], [], ""
        in_git = item.get("inGit")
        sec = item.get("secondary", {})
        def_ok = sec.get("Definition") in GOOD
        if item.get("staging"):
            recovery = "Dataflow Gen2 staging: Fabric recreates it; not saved or restored (V2_4 deletes it)"
        elif t in ITEM_TYPES:
            if t in CHILD_TYPES:
                recovery = "recreated with its Lakehouse/Warehouse; SQL objects from scripts"
            elif in_git:
                recovery = "Git" + (" (after the commit)" if in_git == "after commit" else "")
            elif ws.get("noGit"):
                recovery = "ADLS definition (workspace without Git)"
                if not def_ok:
                    reasons.append("no OK definition in the V2_2 backup")
            elif ALLOW_SECONDARY_ONLY_ITEMS and def_ok:
                recovery = "ADLS definition (V2_5, not in Git)"
            else:
                reasons.append("not saved in Git")
            reasons += [f"secondary {s}: {v or 'missing'}" for s, v in sec.items() if v not in GOOD]
            if item.get("schemaProblem"):
                reasons.append(item["schemaProblem"])
            if t in DATA_TYPES and sec.get(KQL_ADX_STEP) in GOOD:
                recovery += "; data from ADX (Kusto Copy)"
            elif t in DATA_TYPES and sec.get(DATA_TYPES[t]) in GOOD:
                recovery += "; data from ADLS"
        elif t in ACKNOWLEDGED_ITEM_TYPES:
            recovery = "Git (definition only, no data backup)" if in_git else "acknowledged: NOTHING saved"
        else:
            reasons.append(f"type {t} isn't in ITEM_TYPES or ACKNOWLEDGED_ITEM_TYPES. It will be deleted; "
                           + ("Git has its definition, but no data is backed up" if in_git else "nothing is saved"))
        item.update(recovery=recovery, reasons=reasons, status="READY" if not reasons else "BLOCKED")
        for r in reasons:
            log(ws["name"], item, "Gate", "Blocked", r)
        rows.append({"Workspace": ws["name"], "Item": item["displayName"], "Type": t, "Status": item["status"],
                     "Recovery": recovery, "In Git": in_git, "Logical ID": item.get("logicalId", ""),
                     **{f"Secondary {s}": v for s, v in sec.items()}, "Why blocked": "; ".join(reasons)})
    for item in ws["pbi"]:
        rows.append({"Workspace": ws["name"], "Item": item["displayName"], "Type": item["type"], "Status": "LEFT ALONE",
                     "Recovery": "Power BI item: moves with the workspace", "In Git": item.get("inGit")})
    ws["ready"] = not ws["blockers"] and all(i["status"] == "READY" for i in ws["items"])
    ws["status"] = ("READY after commit" if ws["ready"] and ws.get("pendingCommit") else
                    "READY" if ws["ready"] else "BLOCKED")

GATE_DF = pd.DataFrame(rows)
WS_DF = pd.DataFrame([{"Workspace": w["name"], "Workspace ID": w["id"], "Status": w["status"],
                       "Fabric items": len(w["items"]), "Blocked items": sum(i["status"] != "READY" for i in w["items"]),
                       "Repository": w["git"].get("repositoryName"), "Branch": w["git"].get("branchName"),
                       "Commit": w.get("head"), "Workspace blockers": " | ".join(w["blockers"])} for w in WS.values()])
show(WS_DF)
show(GATE_DF.sort_values(["Workspace", "Status", "Type"]), rows=1000)

## 6. Raw copy to ADLS (fallback)
For each READY workspace, with `SAVE_RAW_COPY_TO_ADLS = True` and `DRY_RUN = False`:
- `raw/`: every item's definition exactly as the Fabric API returns it, in the backup layout
  (`<workspace>__<id>/<item>.<Type>__<id>/item.json` + `definition/`), with `backup_log.csv` and run markers;
- `git/`: every file of the snapshot commit, same paths as in the repository, so it can be pushed to any Git repo.

A failed raw copy of an in-scope item blocks the workspace (no snapshot branch, nothing may be deleted).

In [ ]:
RAW_PREFIX = f"{migration_prefix(SNAPSHOT_ID)}/raw"
GIT_MIRROR_PREFIX = f"{migration_prefix(SNAPSHOT_ID)}/git"
raw_log = []


def raw_item_properties(item):
    if item["type"] == "Lakehouse" and item.get("sqlEndpointId"):
        return {"sqlEndpointProperties": {"id": item["sqlEndpointId"]}}
    if item["type"] == "KQLDatabase" and item.get("parentEventhouseItemId"):
        return {"parentEventhouseItemId": item["parentEventhouseItemId"]}
    return {}


def save_raw(ws, item):
    folder = f"{RAW_PREFIX}/{safe_name(ws['name'])}__{ws['id']}/{safe_name(item['displayName'])}.{item['type']}__{item['id']}"
    d = fabric("POST", f"/workspaces/{ws['id']}/items/{item['id']}/getDefinition").get("definition") or {}
    parts = d.get("parts") or []
    if not parts:
        raise HttpError(None, "EmptyDefinition", "the definition came back empty", item["id"])
    size = 0
    for p in parts:
        data = base64.b64decode(p.get("payload") or "")
        size += adls_write(f"{folder}/definition/{p['path']}", data)
    adls_write(f"{folder}/item.json", json.dumps({
        "id": item["id"], "displayName": item["displayName"], "type": item["type"], "description": "",
        "workspaceId": ws["id"], "workspaceName": ws["name"], "properties": raw_item_properties(item),
        "definitionFormat": d.get("format")}, indent=2))
    stored = [n for n, _, is_dir in adls_list(f"{folder}/definition") if not is_dir]
    if len(stored) != len(parts):
        raise HttpError(None, "Verify", f"{len(stored)} of {len(parts)} parts found in ADLS after writing", folder)
    item["rawHash"] = definition_hash(parts)     # V2_4 compares the live definition with it (workspaces without Git)
    return len(parts), size


def save_git_mirror(ws):
    try:
        mirror = f"{GIT_MIRROR_PREFIX}/{safe_name(ws['name'])}__{ws['id']}"
        paths = ado_tree(ws["git"], ws["head"])
        for p in paths:
            adls_write(f"{mirror}/files{p}", ado_file(ws["git"], ws["head"], p))
        adls_write(f"{mirror}/_commit.json", json.dumps({**ws["git"], "commit": ws["head"], "files": len(paths)},
                                                        indent=2))
        ws["gitMirror"] = mirror
        log(ws["name"], {}, "Git file copy", "OK", f"{len(paths)} files of {ws['head']} in {mirror}/files")
    except HttpError as exc:
        block(ws, "Git file copy", exc.short())


if SAVE_RAW_COPY_TO_ADLS and not DRY_RUN:
    for ws in WS.values():
        if not ws["ready"]:
            continue
        for item in ws["items"]:
            if item["type"] in CHILD_TYPES or item.get("staging"):
                continue
            try:
                n, size = save_raw(ws, item)
                item["rawCopy"] = "OK"
                raw_log.append({"Workspace": ws["name"], "WorkspaceId": ws["id"], "Item": item["displayName"],
                                "Type": item["type"], "ItemId": item["id"], "Step": "Definition", "Status": "OK",
                                "Files": n, "Bytes": size, "Rows": "", "Detail": "raw copy from the V2 snapshot"})
            except HttpError as exc:
                item["rawCopy"] = f"Failed: {exc.short()}"
                raw_log.append({"Workspace": ws["name"], "WorkspaceId": ws["id"], "Item": item["displayName"],
                                "Type": item["type"], "ItemId": item["id"], "Step": "Definition", "Status": "Failed",
                                "Files": 0, "Bytes": 0, "Rows": "", "Detail": exc.short()})
                if item["type"] in ITEM_TYPES:
                    item["status"] = "BLOCKED"
                    item["reasons"].append(f"raw copy failed: {exc.short()}")
                    log(ws["name"], item, "Raw copy", "Blocked", exc.short())
                else:
                    log(ws["name"], item, "Raw copy", "Warning", exc.short())
        # file copy of the snapshot commit (workspaces without Git have none)
        if not ws.get("noGit"):
            save_git_mirror(ws)
        if any(i["status"] != "READY" for i in ws["items"]) or ws["blockers"]:
            ws["ready"], ws["status"] = False, "BLOCKED"
            if not ws["blockers"]:
                ws["blockers"].append("Raw copy: failed for at least one in-scope item")
    if raw_log:
        df_raw = pd.DataFrame(raw_log)
        adls_write(f"{RAW_PREFIX}/_run_started.json", json.dumps({"runId": f"_migration/{SNAPSHOT_ID}/raw",
                                                                  "startedUtc": dt.datetime.now(dt.timezone.utc).isoformat(),
                                                                  "kind": "raw definitions from the V2 snapshot"}))
        adls_write(f"{RAW_PREFIX}/backup_log.csv", df_raw.to_csv(index=False))
        adls_write(f"{RAW_PREFIX}/_run_completed.json", json.dumps({
            "runId": f"_migration/{SNAPSHOT_ID}/raw", "finishedUtc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "items": len(df_raw), "problems": int((df_raw.Status != "OK").sum())}))
        print(f"Raw copy: {int((df_raw.Status == 'OK').sum())} of {len(df_raw)} definitions saved to {RAW_PREFIX}/")
    by_key = {(w["name"], i["displayName"], i["type"]): i for w in WS.values() for i in w["items"]}
    GATE_DF["Raw copy"] = [by_key.get((r.Workspace, r.Item, r.Type), {}).get("rawCopy", "") for r in GATE_DF.itertuples()]
    GATE_DF["Status"] = [by_key.get((r.Workspace, r.Item, r.Type), {}).get("status", r.Status) for r in GATE_DF.itertuples()]
elif SAVE_RAW_COPY_TO_ADLS:
    print("Dry run: the raw copy is made when DRY_RUN = False.")

## 7. Snapshot branches and manifest
Creates one locked branch `migration-snapshot/<snapshot id>` (same name for every workspace) and the tag `migration-snapshot-<snapshot id>` at the branch head after all commits, after checking that no workspace's folder changed after its own commit.
Then writes the manifest that the delete and restore notebooks require. Blocked workspaces are recorded but get no
snapshot branch: fix them and run this notebook again (a new snapshot ID).

In [ ]:
SNAPSHOT_BRANCH = f"{SNAPSHOT_BRANCH_PREFIX}/{SNAPSHOT_ID}"
SNAPSHOT_TAG = f"{SNAPSHOT_BRANCH_PREFIX}-{SNAPSHOT_ID}"

if DRY_RUN:
    print("DRY_RUN = True: no commit, no branch, no manifest. Fix the BLOCKED rows, then run with DRY_RUN = False.")
    print(f"Snapshot branch for all workspaces: {SNAPSHOT_BRANCH}" + (f", tag {SNAPSHOT_TAG}" if CREATE_SNAPSHOT_TAG else ""))
    if SYNC_STOP:
        print(f"A real run would stop here too: not every workspace is synced ({', '.join(NOT_SYNCED)}).")
elif SYNC_STOP:
    print(f"NO SNAPSHOT TAKEN: not every workspace is synced with Git ({', '.join(NOT_SYNCED)}). Nothing was "
          "committed, no branch, no raw copy, no manifest. Follow the Fix column of the sync table, then run again.")
else:
    groups = defaultdict(list)                       # one shared branch per repository + working branch
    for ws in WS.values():
        if ws["ready"] and not ws.get("noGit"):
            g = ws["git"]
            groups[(g["organizationName"].lower(), g["projectName"].lower(), g["repositoryName"].lower(),
                    g["branchName"])].append(ws)
    for (_, _, repo, branch_name), members in groups.items():
        gp = members[0]["git"]
        try:
            head = (ado_ref(gp, branch_name) or {}).get("objectId")
            if not head:
                raise HttpError(None, "NoBranch", f"branch {branch_name} not found", branch_name)
            # Every workspace's folder must be identical at the shared head and at its own commit: nothing changed
            # in its folder after it was committed.
            for ws in members:
                if ws["head"] != head and ado_tree_ids(ws["git"], ws["head"]) != ado_tree_ids(ws["git"], head):
                    ws["ready"], ws["status"] = False, "BLOCKED"
                    block(ws, "Snapshot branch", f"its Git folder changed after its commit ({branch_name} is now at "
                          f"{head}): take a new snapshot")
            members = [ws for ws in members if ws["ready"]]
            if not members:
                continue
            if ado_ref(gp, SNAPSHOT_BRANCH):
                raise HttpError(None, "BranchExists", f"branch {SNAPSHOT_BRANCH} already exists in {repo}", repo)
            ado_create_branch(gp, SNAPSHOT_BRANCH, head)
            if LOCK_SNAPSHOT_BRANCHES:
                ado_lock(gp, SNAPSHOT_BRANCH)
            ref = ado_ref(gp, SNAPSHOT_BRANCH) or {}
            if ref.get("objectId") != head or (LOCK_SNAPSHOT_BRANCHES and not ref.get("isLocked")):
                raise HttpError(None, "Verify", f"branch {SNAPSHOT_BRANCH} isn't at {head} and locked", repo)
        except HttpError as exc:
            for ws in members:
                ws["ready"], ws["status"] = False, "BLOCKED"
                block(ws, "Snapshot branch", f"{exc.short()}. The token needs Create branch and Contribute.")
            continue
        tag = ""
        if CREATE_SNAPSHOT_TAG:
            try:
                ado_create_tag(gp, SNAPSHOT_TAG, head, f"Fabric migration snapshot {SNAPSHOT_ID}: "
                               + ", ".join(w["name"] for w in members))
                tag = SNAPSHOT_TAG
            except HttpError as exc:
                for ws in members:
                    log(ws["name"], {}, "Snapshot tag", "Warning", f"{exc.short()} (the locked branch still holds "
                        "the snapshot; tags need the Create tag permission)")
        for ws in members:
            ws["ownCommit"], ws["head"] = ws["head"], head
            ws["snapshotBranch"], ws["snapshotTag"] = SNAPSHOT_BRANCH, tag
            log(ws["name"], {}, "Snapshot branch", "OK", f"{SNAPSHOT_BRANCH} at {head}"
                + (" (locked)" if LOCK_SNAPSHOT_BRANCHES else "") + (f", tag {tag}" if tag else ""))

    keep = ("id", "type", "displayName", "logicalId", "gitFolder", "gitFiles", "inGit", "sqlEndpointId",
            "parentEventhouseItemId", "defaultSchema", "staging", "secondary", "rawCopy", "rawHash", "recovery", "status", "reasons")
    MANIFEST = {
        "snapshotId": SNAPSHOT_ID, "createdUtc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "itemTypes": ITEM_TYPES, "acknowledgedItemTypes": ACKNOWLEDGED_ITEM_TYPES,
        "secondaryBackup": {"account": ADLS_ACCOUNT, "container": ADLS_CONTAINER, "root": ADLS_ROOT,
                            "runId": SECONDARY_BACKUP_RUN_ID, "startedUtc": run_started,
                            "freezeStartedUtc": FREEZE_STARTED_UTC},
        "rawCopy": {"enabled": SAVE_RAW_COPY_TO_ADLS, "definitionsRunId": f"_migration/{SNAPSHOT_ID}/raw",
                    "gitMirror": f"_migration/{SNAPSHOT_ID}/git"} if SAVE_RAW_COPY_TO_ADLS else {"enabled": False},
        "workspaces": [{
            "id": w["id"], "name": w["name"], "capacityId": w["capacityId"], "ready": w["ready"],
            "gitRoute": "none" if w.get("noGit") else "git",
            "blockers": w["blockers"], "secondaryFolder": w.get("secondaryFolder", ""),
            "gitMirror": w.get("gitMirror", ""),
            "git": {**w["git"], "commit": w.get("head"), "snapshotBranch": w.get("snapshotBranch"),
                    "snapshotTag": w.get("snapshotTag", ""), "workspaceCommit": w.get("ownCommit", w.get("head")),
                    "locked": bool(w.get("snapshotBranch")) and LOCK_SNAPSHOT_BRANCHES},
            "items": [{k: i.get(k) for k in keep} for i in w["items"]],
            "powerBiItems": [{k: i.get(k) for k in ("id", "type", "displayName", "inGit", "gitFolder")} for i in w["pbi"]],
        } for w in WS.values()]}
    prefix = migration_prefix(SNAPSHOT_ID)
    raw = json.dumps(MANIFEST, indent=2, sort_keys=True, default=str).encode()
    adls_write(f"{prefix}/manifest.json", raw)
    adls_write(f"{prefix}/manifest.sha256", hashlib.sha256(raw).hexdigest())
    adls_write(f"{prefix}/items.csv", GATE_DF.to_csv(index=False))
    save_log(prefix, "snapshot_log")
    ready = [w["name"] for w in WS.values() if w["ready"]]
    print(f"\nSNAPSHOT ID: {SNAPSHOT_ID}\nSnapshot branch (all workspaces): {SNAPSHOT_BRANCH}"
          + (f", tag {SNAPSHOT_TAG}" if CREATE_SNAPSHOT_TAG else ""))
    print(f"Manifest: abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net/{prefix}/manifest.json")
    print(f"READY ({len(ready)}): {', '.join(ready) or '-'}")
    blocked = [w["name"] for w in WS.values() if not w["ready"]]
    if blocked:
        print(f"BLOCKED ({len(blocked)}): {', '.join(blocked)}. Don't delete anything in these workspaces.")
    print("\nFrom now on, no changes in these workspaces until the restore is verified. Next: V2_4_Delete_Fabric_Items "
          "with this SNAPSHOT ID.")

## After this notebook
- Keep the change freeze. Anything changed after the snapshot isn't in Git or ADLS; the delete notebook checks
  this and refuses.
- The snapshot branches stay locked. Don't delete them until the migration is signed off.
- If a workspace is BLOCKED, fix the reason, take a new secondary backup if data changed, and run this notebook again.